In [31]:
import os
import re
import json
import math

from IPython.display import JSON

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv

from google import genai
from google.genai import types
from ydata_profiling import ProfileReport


In [32]:
df = pd.read_csv(r"C:\Users\shubh\Desktop\Auto EDA\My work\Zomato Restaurant Dataset.csv")

def get_dataframe_metadata(df):

    metadata = {
        "row_count": len(df),
        "column_count": len(df.columns),
        "columns": {}
    }

    for column in df.columns:

        metadata["columns"][column] = {
            "dtype": str(df[column].dtype),
            "missing_count": int(df[column].isna().sum()),
            "unique_count": int(df[column].nunique()),
            "sample_values": (
                df[column]
                .dropna()
                .head(5)
                .tolist()
            )
        }

    return metadata

metadata = get_dataframe_metadata(df)

In [33]:
def get_analysis_questions(report_path, metadata_path):
    
    with open(report_path, "r", encoding="utf-8") as f:
        mod_report = json.load(f)

    prompt = f"""
    You are a senior data analyst. The dataset is already cleaned.
    Your goal is to find potential trends in this data. Do NOT summarize
    the dataset or describe the report. Only propose questions worth
    asking and the aggregations that would answer them.

    Use the metadata to understand what each column means, and the
    profiling report to see which columns are usable (cardinality,
    distribution, date ranges, correlations).

    First, classify columns internally as:
    - Dimensions (categories to group by, e.g. country, product, segment)
    - Measures (numeric values to aggregate, e.g. revenue, count, price)
    - Time columns (dates or periods)

    Then generate analysis questions in these categories:

    1. Aggregation by a dimension
       (e.g. total or average of a measure by country)
    2. Drill-down within a dimension
       (e.g. within a country, break down by region, product, or segment)
    3. Trends over time
       (e.g. monthly or yearly change in a measure, overall or per group)
    4. Comparisons across groups
       (e.g. top vs bottom performers, share of total, ranking changes)
    5. Relationships between columns
       (e.g. does measure A change with measure B or a category)
    6. Concentration and outliers
       (e.g. do a few groups drive most of the total, which groups
       behave unlike the rest)

    For each question, give:
    - The question, in plain language
    - Columns involved
    - The aggregation (sum, mean, median, count, ratio, growth rate)
      and the grouping level
    - What kind of trend or pattern it could reveal

    Rules:
    - Only use columns that exist in the metadata or report.
    - Skip columns that are unsuitable (e.g. IDs, near-constant columns,
      very high-cardinality categories for grouping).
    - Prefer questions where the report hints at something interesting
      (skew, strong correlation, imbalance, wide date range).
    - Do not write Python code or run any analysis.
    - Do not recommend data cleaning.
    - Do not claim a trend exists. Phrase everything as a question
      to investigate.
    - Give 10 to 15 questions, ordered from most to least promising.

    Metadata:
    {metadata}

    Profiling report:
    {json.dumps(mod_report)}
    """

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )

    return response.text

In [34]:
# Example

analysis_questions = get_analysis_questions("mod_report.json", metadata)

print(analysis_questions)

Here's an analysis plan with questions and aggregations to find potential trends in the restaurant data.

---

**Column Classification:**

*   **Dimensions:** `Country Code`, `City`, `Locality`, `Cuisines`, `Currency`, `Has Table booking`, `Has Online delivery`, `Price range`, `Rating color`, `Rating text`.
*   **Measures:** `Average Cost for two`, `Aggregate rating`, `Votes`.
*   **Time columns:** None identified in the profiling report (`"time_index_analysis": "None"`), so no questions for this category.

**Analysis Questions:**

1.  **Question:** What are the top 5 countries with the highest average restaurant ratings?
    *   **Columns involved:** `Country Code`, `Aggregate rating`
    *   **Aggregation:** Mean of `Aggregate rating`, grouped by `Country Code`.
    *   **Trend/Pattern:** Identifies regions with overall higher customer satisfaction or service quality.

2.  **Question:** Which cities have the highest concentration of restaurants, and how does this relate to their aver

In [35]:


def generate_analysis_functions(analysis_questions, df_metadata):

    prompt = f"""
    You are a Python data-analysis function generator.

    INPUT 1: ANALYSIS QUESTIONS
    {analysis_questions}

    INPUT 2: DATAFRAME METADATA
    {json.dumps(df_metadata, default=str)}

    YOUR TASK:
    Generate one small Python function per analysis question.
    Each function computes the aggregation that answers the
    question and returns the result as a DataFrame.

    RULES:
    1. Generate one function per question. Skip questions that
       cannot be answered with the available columns.
    2. Use the exact DataFrame column names provided.
    3. Each function must accept df as its only argument.
    4. Each function must return a pandas DataFrame (the result
       table), never a plain number, string, or plot.
    5. Do not modify the original DataFrame. Work on a copy.
    6. Do not invent columns or assume unsupported facts.
    7. Only pandas (pd) is available. Do not include imports,
       plotting, file I/O, or print statements.
    8. Do not clean data. Assume the DataFrame is already clean.
    9. For time trends, convert date columns inside the function
       only if needed, and group by period (month, quarter, year).
    10. For drill-downs, group by the parent dimension first,
        then the child dimension.
    11. Keep results readable: sort sensibly, reset the index,
        and limit very long outputs with .head(20) where it makes
        sense (e.g. top groups).
    12. Where useful, add derived columns such as share_of_total
        or growth_rate to the result.
    13. Do not use Markdown code fences.

    RETURN FORMAT:
    Return valid JSON with this structure:
    {{
        "functions": [
            {{
                "name": "function_name",
                "question": "the analysis question in plain language",
                "columns": ["column_name"],
                "group_by": ["column_name"],
                "aggregation": "sum / mean / count / growth_rate etc.",
                "code": "complete Python function as a string"
            }}
        ]
    }}

    Return an empty functions list if no question can be
    answered safely with the available columns.
    """

    response = client.models.generate_content(
        model="gemini-3.7-flash",
        contents=prompt,
        config=types.GenerateContentConfig(
            temperature=0.1,
            response_mime_type="application/json"
        )
    )

    return response.text

In [36]:
import pandas as pd
from IPython.display import display

response_data = json.loads(generate_analysis_functions(analysis_questions, metadata))

results = {}

for func_data in response_data["functions"]:
    function_name = func_data["name"]

    try:
        namespace = {"pd": pd}
        exec(func_data["code"], namespace)

        analysis_function = namespace[function_name]
        result_df = analysis_function(df.copy())
        results[function_name] = result_df

        print(f"\n{'=' * 80}")
        print(f"{function_name}")
        print(f"Question: {func_data['question']}")
        print(f"{'=' * 80}")
        display(result_df)

    except Exception as e:
        print(f"\n[FAILED] {function_name} — {type(e).__name__}: {e}")


top_countries_by_rating
Question: What are the top 5 countries with the highest average restaurant ratings?


,Country Code,mean_aggregate_rating
0,162,4.468182
1,208,4.300000
2,94,4.295238
3,148,4.262500
4,214,4.233333



restaurant_density_and_rating_by_city
Question: Which cities have the highest concentration of restaurants, and how does this relate to their average aggregate rating?


,City,restaurant_count,mean_aggregate_rating
0,New Delhi,5473,2.438845
1,Gurgaon,1118,2.651431
2,Noida,1080,2.036204
3,Faridabad,251,1.866932
4,Ghaziabad,25,2.852000
5,Bhubaneshwar,21,3.980952
6,Lucknow,21,4.195238
7,Guwahati,21,4.190476
8,Ahmedabad,21,4.161905
9,Amritsar,21,3.685714



prevalent_cuisines_and_ratings
Question: What are the most prevalent cuisine types offered by restaurants, and how do their average ratings compare?


...

,Rating color,Price range,restaurant_count
0,Dark Green,1,32
1,Dark Green,2,69
2,Dark Green,3,126
3,Dark Green,4,74
4,Green,1,144
5,Green,2,325
6,Green,3,416
7,Green,4,194
8,Orange,1,1898
9,Orange,2,1425



online_delivery_by_currency
Question: What is the ratio of restaurants offering online delivery in countries where Indian Rupee is the currency versus other currencies?


,Currency,total_restaurants,delivery_restaurants,online_delivery_ratio,delivery_percentage
0,Indian Rupees(Rs.),8652,2423,0.280051,28.005086
1,Dollar($),482,0,0.000000,0.000000
2,Pounds(��),80,0,0.000000,0.000000
3,Brazilian Real(R$),60,0,0.000000,0.000000
4,Rand(R),60,0,0.000000,0.000000
5,Emirati Diram(AED),60,28,0.466667,46.666667
6,NewZealand($),40,0,0.000000,0.000000
7,Turkish Lira(TL),34,0,0.000000,0.000000
8,Botswana Pula(P),22,0,0.000000,0.000000
9,Indonesian Rupiah(IDR),21,0,0.000000,0.000000



zero_coordinates_by_city
Question: Are there any specific cities where Longitude or Latitude values are frequently zero, potentially indicating missing or default geographic data?


,City,total_restaurants,zero_coord_count,zero_coord_pct
0,New Delhi,5473,233,4.257263
1,Noida,1080,94,8.703704
2,Gurgaon,1118,48,4.293381
3,Faridabad,251,22,8.764940
4,Vadodara,20,10,50.000000
5,Mangalore,20,9,45.000000
6,Dehradun,20,7,35.000000
7,Varanasi,20,7,35.000000
8,Bhopal,20,6,30.000000
9,Nashik,20,6,30.000000
